In [13]:
pip install datasets

In [14]:
import pandas as pd
import numpy as np
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
import spacy
import os

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sklearn import metrics

from sklearn.metrics import classification_report

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

from string import punctuation
from collections import Counter
import random

import torch
import torch.nn as nn
from transformers import BertModel, BertConfig

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [15]:
def set_random_seed(seed):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
seed = 42
set_random_seed(seed)

In [16]:
RANDOM_SEED = 12345

# Подготовка данных

In [17]:
df_alex = pd.read_csv(r'/content/dataset_all_data.csv')

In [18]:
# Общая информация и пропуски
print("Общая информация по DataFrame:")
print(df_alex.info())
print("\nКоличество пропусков по столбцам:")
print(df_alex.isna().sum())

# Распределение сарказма
print("\nРаспределение меток sarcasm:")
print(df_alex['sarcasm'].value_counts(dropna=False))
print("\nДоля каждой метки sarcasm:")
print(df_alex['sarcasm'].value_counts(normalize=True))


# Статистика по длине текста
df_alex['text_length_chars'] = df_alex['text'].str.len()
df_alex['text_length_words'] = df_alex['text'].str.split().str.len()

print("\nСтатистика длины текста (символы):")
print(df_alex['text_length_chars'].describe())
print("\nСтатистика длины текста (слова):")
print(df_alex['text_length_words'].describe())

# Сравнение длины текста в саркастичных/несаркастичных примерах
print("\nДлина текста по классам sarcasm (символы):")
print(df_alex.groupby('sarcasm')['text_length_chars'].describe())

print("\nДлина текста по классам sarcasm (слова):")
print(df_alex.groupby('sarcasm')['text_length_words'].describe())

Общая информация по DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9144 entries, 0 to 9143
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   text     9144 non-null   object
 1   gender   9144 non-null   int64 
 2   age      9144 non-null   int64 
 3   sarcasm  9144 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 285.9+ KB
None

Количество пропусков по столбцам:
text       0
gender     0
age        0
sarcasm    0
dtype: int64

Распределение меток sarcasm:
sarcasm
0    7620
1    1524
Name: count, dtype: int64

Доля каждой метки sarcasm:
sarcasm
0    0.833333
1    0.166667
Name: proportion, dtype: float64

Статистика длины текста (символы):
count    9144.000000
mean      134.554462
std        80.719181
min         6.000000
25%        79.000000
50%       105.000000
75%       162.000000
max       717.000000
Name: text_length_chars, dtype: float64

Статистика длины текста (слова):
count    9144.000000
mean

# Сашин корпус

## Обучение моделей

In [19]:
y = df_alex['sarcasm']
X = df_alex.drop('sarcasm', axis=1)

In [20]:
def feature_engineering(choice_transformer, choice_ngrams):
    # Обработка текстовых данных: либо TF-IDF, либо мешок слов
    text_features = 'text'
    if choice_transformer == 'tfidf':
        text_transformer = TfidfVectorizer(
            ngram_range=choice_ngrams,
            tokenizer=word_tokenize,
            stop_words='english'
        )
    else:
        text_transformer = CountVectorizer(
            ngram_range=choice_ngrams,
            tokenizer=word_tokenize,
            stop_words='english'
        )

    # Применяем трансформацию только к текстовому столбцу 'text'
    preprocessor = ColumnTransformer(
        transformers=[
            ("txt", text_transformer, text_features)
        ]
    )
    return preprocessor

In [21]:
def modelfit(model):
    model.fit(Xtrain, ytrain)

    # Предсказания меток
    ypredtest = model.predict(Xtest)
    ypredtrain = model.predict(Xtrain)

    # Предсказания вероятностей (для roc_auc_score)
    yprobtest = model.predict_proba(Xtest)
    yprobtrain = model.predict_proba(Xtrain)

    print('RESULTS:\nroc-auc_score:\n',
          #'train:', roc_auc_score(ytrain, yprobtrain, multi_class='ovr'),
          #'test:', roc_auc_score(ytest, yprobtest, multi_class='ovr'),
          '\nf1_score:\n',
          'train:', f1_score(ytrain, ypredtrain, average='macro'),
          'test:', f1_score(ytest, ypredtest, average='macro'),
          '\nclassification_report\ntrain:\n',
          classification_report(ytrain, ypredtrain),
          '\ntest:\n',
          classification_report(ytest, ypredtest)
         )

In [22]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED)

In [23]:
ytrain = ytrain.squeeze()
ytest = ytest.squeeze()

In [24]:
preprocessor = feature_engineering('tfidf', (1, 1))

clfLR = Pipeline(
    steps=[("preprocessor", preprocessor), ("classifier", LogisticRegression(class_weight='balanced', random_state = RANDOM_SEED))]
)

clfSVC = Pipeline(
    steps=[("preprocessor", preprocessor), ("classifier", SVC(class_weight='balanced', probability=True, random_state = RANDOM_SEED))]
)

### Логистическая регрессия

In [25]:
modelfit(clfLR)

/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


RESULTS:
roc-auc_score:
 
f1_score:
 train: 0.8874361041766807 test: 0.6345153424417329 
classification_report
train:
               precision    recall  f1-score   support

           0       0.99      0.92      0.96      6096
           1       0.71      0.98      0.82      1219

    accuracy                           0.93      7315
   macro avg       0.85      0.95      0.89      7315
weighted avg       0.95      0.93      0.93      7315
 
test:
               precision    recall  f1-score   support

           0       0.90      0.80      0.84      1524
           1       0.35      0.54      0.43       305

    accuracy                           0.75      1829
   macro avg       0.62      0.67      0.63      1829
weighted avg       0.81      0.75      0.77      1829



### Метод опорных векторов

In [26]:
modelfit(clfSVC)

/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


RESULTS:
roc-auc_score:
 
f1_score:
 train: 0.999262435969813 test: 0.5809653830461664 
classification_report
train:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      6096
           1       1.00      1.00      1.00      1219

    accuracy                           1.00      7315
   macro avg       1.00      1.00      1.00      7315
weighted avg       1.00      1.00      1.00      7315
 
test:
               precision    recall  f1-score   support

           0       0.85      0.98      0.91      1524
           1       0.58      0.16      0.25       305

    accuracy                           0.84      1829
   macro avg       0.71      0.57      0.58      1829
weighted avg       0.81      0.84      0.80      1829



### Базовый Берт

In [27]:
class CustomBertClassifier(nn.Module):
    def __init__(self,
                 pretrained_model_name: str = 'DeepPavlov/rubert-base-cased',
                 num_labels: int = 2,
                 hidden_dim: int = 768,
                 dropout_prob: float = 0.1):
        super().__init__()
        # Базовая модель берта без головы для маскированного языка
        self.bert = BertModel.from_pretrained(pretrained_model_name)

        # Кастомная голова
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_prob),
            nn.Linear(self.bert.config.hidden_size, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            nn.Linear(hidden_dim, num_labels)
        )

    def forward(self,
                input_ids: torch.LongTensor,
                attention_mask: torch.Tensor = None,
                token_type_ids: torch.Tensor = None,
                labels: torch.LongTensor = None):
        # Получаем выходы из Берт
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True
        )
        # Выбираем pooled_output
        pooled_output = outputs.pooler_output

        # Передаём через свою голову
        logits = self.classifier(pooled_output)

        # Если есть метки - считаем loss
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
            return {
                'loss': loss,
                'logits': logits
            }
        return {'logits': logits}

In [28]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset as TorchDataset, DataLoader
from transformers import BertTokenizer, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from dataclasses import dataclass

# Загружаем и обрабатываем
df_alex['sarcasm'] = df_alex['sarcasm'].astype(int)

train_df, test_df = train_test_split(
    df_alex,
    test_size=0.2,
    stratify=df_alex['sarcasm'],
    random_state=RANDOM_SEED
)

# Токенизация
tokenizer = BertTokenizer.from_pretrained('DeepPavlov/rubert-base-cased')

@dataclass
class NERFeatures:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    token_type_ids: torch.Tensor
    labels: torch.Tensor

class SarcasmDataset(TorchDataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df['text'].tolist()
        self.labels = df['sarcasm'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        enc = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'token_type_ids': enc.get('token_type_ids', torch.zeros_like(enc['input_ids'])).squeeze(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Создаём датасеты
train_dataset = SarcasmDataset(train_df, tokenizer)
test_dataset  = SarcasmDataset(test_df, tokenizer)

# Загрузка модели
model = CustomBertClassifier(
    pretrained_model_name='DeepPavlov/rubert-base-cased',
    num_labels=2,
    hidden_dim=768,
    dropout_prob=0.1
)

# Метрики
def compute_metrics(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='macro')
    }

# Параметры тренировки
training_args = TrainingArguments(
    output_dir="./results",
    do_train=True,
    do_eval=True,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=2,

    # частота в шагах
    logging_steps=50,
    eval_steps=500,
    save_steps=500,
    save_total_limit=1,

    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
)

# Трейнер
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# Обучение и оценка
trainer.train()
results = trainer.evaluate()
print(results)
# Сохраняем лучшую модель
trainer.save_model('./best_model')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Step,Training Loss
50,0.481200
100,0.428500
150,0.414000
200,0.396000
250,0.361600
300,0.320500
350,0.327900
400,0.316300
450,0.275200
500,0.243100


{'eval_loss': 0.4120042026042938, 'eval_accuracy': 0.8485511208310552, 'eval_f1': 0.6939219122774267, 'eval_runtime': 3.0115, 'eval_samples_per_second': 607.329, 'eval_steps_per_second': 9.63, 'epoch': 3.0}
